In [ ]:
import importlib
import importlib.util
import site
import subprocess
import sys

# install only if missing
if importlib.util.find_spec("pydantic") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pydantic<=1.10.13"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    importlib.reload(site)

In [ ]:
import io
import json
import logging
import os
import shutil
import subprocess
import threading
import time
import typing as tp
from enum import Enum
from pathlib import Path

import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from pydantic import BaseModel, root_validator, validator
from traitlets import Bunch

In [ ]:
# Only install if not already on PATH
if shutil.which("uv") is None:
    subprocess.run(
        ["bash", "-c", "wget -qO- https://astral.sh/uv/install.sh | sh"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    # print("✔️ uv installed successfully")
else:
    pass
    # print("✔️ uv already installed")

In [ ]:
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.local/bin")

REPO_DIR_NAME = "SAR-Pattern-Validation"

WORKSPACE_ROOT = Path.cwd().resolve()
PROJECT_ROOT = WORKSPACE_ROOT / REPO_DIR_NAME

# The Makefile ships with the oSPARC service at the workspace top level.
# It handles cloning/pulling the repo, installing git-lfs, and fetching LFS files.
subprocess.run(["make", "setup"])


def run_sarsample(*args):
    cmd = [
        "uvx",
        "--no-cache",
        "--from",
        str(PROJECT_ROOT),
        "sar-pattern-validation",
        *args,
    ]

    # Override MPLBACKEND so the subprocess doesn't inherit the Jupyter inline
    # backend, which is only valid inside a kernel and causes matplotlib to crash.
    env = os.environ.copy()
    env["MPLBACKEND"] = "agg"

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        env=env,
    )
    raw_output = result.stdout if result.returncode == 0 else result.stderr

    try:
        return json.loads(raw_output)
    except json.JSONDecodeError as error:
        raise RuntimeError(
            "sar-pattern-validation did not return valid JSON.\n"
            f"Command: {' '.join(cmd)}\n"
            f"Return code: {result.returncode}\n"
            f"Stdout:\n{result.stdout}\n"
            f"Stderr:\n{result.stderr}"
        ) from error

In [ ]:
class OutputWidgetHandler(logging.Handler):
    """Custom logging handler sending logs to an output widget"""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        layout = {
            "width": "90%",
            "height": "600px",
            "border": "1px solid black",
            "overflow": "scroll hidden",
            "flex_flow": "column",
            "display": "flex",
        }
        self.out = widgets.Output(layout=layout)

    def emit(self, record):
        """Overload of logging.Handler method"""
        formatted_record = self.format(record)
        new_output = {
            "name": "stdout",
            "output_type": "stream",
            "text": formatted_record + "\n",
        }
        self.out.outputs = (new_output,) + self.out.outputs

    def show_logs(self) -> widgets.Output:
        """Show the logs"""
        return self.out

    def clear_logs(self):
        """Clear the current logs"""
        self.out.clear_output()

In [ ]:
class FilePaths(Enum):
    DATABASE_PATH = str(PROJECT_ROOT / "data" / "database")
    NO_DATA_IMAGE = str(PROJECT_ROOT / "assets" / "no-data-transparent.png")
    MEASURED_FILE_PATH = str(WORKSPACE_ROOT / "uploaded_data" / "measured_data.csv")
    REFERENCE_IMAGE_PATH = str(WORKSPACE_ROOT / "images" / "reference_image.png")
    MEASURED_IMAGE_PATH = str(WORKSPACE_ROOT / "images" / "measured_image.png")
    ALIGNED_MEANS_PATH = str(WORKSPACE_ROOT / "images" / "aligned_means_image.png")
    ALIGNED_MEANS_COLORBAR_PATH = str(
        WORKSPACE_ROOT / "images" / "aligned_means_image_colorbar.png"
    )
    REGISTERED_IMAGE_PATH = str(WORKSPACE_ROOT / "images" / "registered_image.png")
    GAMMA_COMPARISON_PATH = str(
        WORKSPACE_ROOT / "images" / "gamma_comparison_image.png"
    )
    GAMMA_COMPARISON_COLORBAR_PATH = str(
        WORKSPACE_ROOT / "images" / "gamma_comparison_image_colorbar_vertical.png"
    )
    GAMMA_COMPARISON_FAILURES_PATH = str(
        WORKSPACE_ROOT / "images" / "gamma_comparison_image_failures.png"
    )
    STATE_JSON_PATH = str(WORKSPACE_ROOT / "system_state" / "state.json")
    WORKFLOW_RESULTS_JSON_PATH = str(
        WORKSPACE_ROOT / "system_state" / "workflow_results.json"
    )
    FILTERED_DB_CSV_PATH = str(WORKSPACE_ROOT / "system_state" / "filtered_db.csv")

In [ ]:
class DBColumnNames(Enum):
    ANTENNA_TYPE = "Antenna Type"
    FREQUENCY = "Frequency [MHz]"
    DISTANCE = "Distance [mm]"
    MASS = "Mass [g]"
    FILE_PATH = "File Path"

In [ ]:
class FilterOption(BaseModel):
    column_name: DBColumnNames
    value: tp.Any

    @validator("column_name")
    def validate_column(cls, v: DBColumnNames) -> DBColumnNames:
        if v == DBColumnNames.FILE_PATH:
            raise ValueError("FILE_PATH cannot be used as a filter column")
        return v

    @validator("value")
    def validate_value(cls, v: tp.Any) -> tp.Any:
        if v is None:
            raise ValueError("Filter value cannot be None")
        return v

    @property
    def column(self) -> str:
        return self.column_name.value


class FilterOptions(BaseModel):
    antenna_type: tp.Optional[FilterOption] = None
    frequency: tp.Optional[FilterOption] = None
    distance: tp.Optional[FilterOption] = None
    mass: tp.Optional[FilterOption] = None

    @root_validator
    def validate_columns(cls, values: dict[str, tp.Any]) -> dict[str, tp.Any]:

        mapping = {
            "antenna_type": DBColumnNames.ANTENNA_TYPE,
            "frequency": DBColumnNames.FREQUENCY,
            "distance": DBColumnNames.DISTANCE,
            "mass": DBColumnNames.MASS,
        }

        for field, expected_column in mapping.items():
            option = values.get(field)

            if option is None:
                continue

            if option.column_name != expected_column:
                raise ValueError(
                    f"{field} filter must use column {expected_column.value}"
                )

        return values

    def iter_active_filters(self) -> tp.Generator[FilterOption, None, None]:
        for option in self.__dict__.values():
            if option is not None:
                yield option

In [ ]:
class SimulatedFilesDB:
    def __init__(self):
        self.simulated_files_db: pd.DataFrame
        self.create_simulated_files_db(path_to_db=Path(FilePaths.DATABASE_PATH.value))
        self.unique_entries_in_columns: dict[str, list[tp.Any]]
        self.unique_entries_in_columns = self.get_unique_values_in_columns(
            files_db=self.simulated_files_db
        )

    def create_simulated_files_db(self, path_to_db: Path) -> None:
        all_file_paths = sorted(path_to_db.glob("**/*.csv"))

        rows: list[dict[str, tp.Any]] = []
        for file_path in all_file_paths:
            parts = file_path.stem.split("_")
            parts.remove("Flat")
            if len(parts) != 4:
                raise ValueError(f"Unexpected filename format: {file_path.name}")

            antenna_type, frequency, distance, mass = parts

            row: dict[str, tp.Any] = {
                DBColumnNames.ANTENNA_TYPE.value: antenna_type,
                DBColumnNames.FREQUENCY.value: float(frequency.replace("MHz", "")),
                DBColumnNames.DISTANCE.value: float(distance.replace("mm", "")),
                DBColumnNames.MASS.value: float(mass.replace("g", "")),
                DBColumnNames.FILE_PATH.value: str(file_path),
            }

            rows.append(row)

        self.simulated_files_db = pd.DataFrame(
            rows,
            columns=[c.value for c in DBColumnNames],
        )

    def get_unique_values_in_columns(
        self, files_db: pd.DataFrame
    ) -> dict[str, list[tp.Any]]:
        unique_entries_in_columns: dict[str, list[tp.Any]] = {}
        for column_name, column in files_db.items():
            if column_name == DBColumnNames.FILE_PATH.value:
                continue
            unique_entries = [x for x in column.unique() if x is not np.nan]
            unique_entries_in_columns[str(column_name)] = sorted(unique_entries)
        return unique_entries_in_columns

    def get_rows_with_filter(self, filter_setup: FilterOptions) -> pd.DataFrame:
        filtered_df: pd.DataFrame = self.simulated_files_db.copy()
        for f in filter_setup.iter_active_filters():
            filtered_df = filtered_df[filtered_df[f.column] == f.value]  # type: ignore
        return filtered_df  # type: ignore

    def save_filtered_db_to_csv(
        self, filtered_df: pd.DataFrame, path: tp.Union[str, Path]
    ) -> None:
        """Save filtered dataframe to CSV file."""
        file_path = Path(path)
        file_path.parent.mkdir(exist_ok=True, parents=True)
        filtered_df.to_csv(file_path, index=False)

    def load_filtered_db_from_csv(self, path: tp.Union[str, Path]) -> pd.DataFrame:
        """Load filtered dataframe from CSV file."""
        return pd.read_csv(path)

In [ ]:
class RadioButtonGrid:
    FILTER_ATTR = {
        DBColumnNames.ANTENNA_TYPE: "antenna_type",
        DBColumnNames.FREQUENCY: "frequency",
        DBColumnNames.DISTANCE: "distance",
        DBColumnNames.MASS: "mass",
    }

    def __init__(self, simulated_files_db: SimulatedFilesDB):
        self.simulated_files_db = simulated_files_db
        self.filter_options = FilterOptions()
        self.filtered_db: pd.DataFrame

        # store buttons per column
        self.button_groups: dict[DBColumnNames, list[widgets.ToggleButton]] = {}

    def create_radio_button_grid(self) -> widgets.HBox:
        all_button_columns: list[widgets.VBox] = []

        for (
            column_name,
            unique_values,
        ) in self.simulated_files_db.unique_entries_in_columns.items():
            try:
                column_enum = [x for x in DBColumnNames if x.value == column_name][0]
            except IndexError:
                print(f"{column_name} not found in DBColumnNames")
                continue
            if column_enum == DBColumnNames.FILE_PATH:
                continue

            buttons: list[widgets.ToggleButton] = []
            for value in unique_values:
                btn = widgets.ToggleButton(
                    description=str(value),
                    disabled=False,
                    layout=widgets.Layout(
                        width="100%",
                    ),
                )
                btn.observe(
                    self._make_handler(btn, column_enum),
                    "value",
                )
                buttons.append(btn)

            self.button_groups[column_enum] = buttons
            column_widget = widgets.VBox(
                [widgets.HTML(f"<b>{column_name}</b>"), *buttons],
                layout=widgets.Layout(
                    flex="1 1 0%",
                    min_width="0",
                    overflow="hidden",
                ),
            )
            all_button_columns.append(column_widget)

        return widgets.HBox(
            all_button_columns,
            layout=widgets.Layout(width="100%"),
        )

    def _make_handler(
        self, button: widgets.ToggleButton, column_enum: DBColumnNames
    ) -> tp.Callable[[Bunch], None]:
        def handler(change: Bunch) -> None:
            if change["name"] != "value":
                return
            column_attr = self.FILTER_ATTR[column_enum]
            value = change.owner.description
            if column_enum != DBColumnNames.ANTENNA_TYPE:
                value = float(value)

            group_buttons = self.button_groups[column_enum]

            # button activated
            if change["new"]:
                # ensure only one button active per column
                for b in group_buttons:
                    if b is not button:
                        b.value = False
                setattr(
                    self.filter_options,
                    column_attr,
                    FilterOption(
                        column_name=column_enum,
                        value=value,
                    ),
                )

            # button deactivated
            else:
                current_filter = getattr(self.filter_options, column_attr)
                if current_filter and current_filter.value == value:
                    setattr(self.filter_options, column_attr, None)
            self.filtered_db = self.simulated_files_db.get_rows_with_filter(
                filter_setup=self.filter_options
            )
            self.update_button_states(self.filtered_db)

        return handler

    def update_button_states(self, filtered_df: pd.DataFrame) -> None:
        for column_enum, buttons in self.button_groups.items():
            column_name = column_enum.value
            valid_values = set(filtered_df[column_name].astype(str))
            for b in buttons:
                if b.description in valid_values:
                    b.disabled = False
                else:
                    b.disabled = True

    def restore_from_filtered_db(self, filtered_df: pd.DataFrame) -> None:
        """Restore button states from a filtered dataframe."""
        # Restore filter_options based on the filtered_df
        self.filtered_db = filtered_df

        # Determine which filters are active by comparing filtered_df to full db
        for column_enum in [
            DBColumnNames.ANTENNA_TYPE,
            DBColumnNames.FREQUENCY,
            DBColumnNames.DISTANCE,
            DBColumnNames.MASS,
        ]:
            column_name = column_enum.value
            unique_values = filtered_df[column_name].unique()

            # If only one unique value, that column is filtered
            if len(unique_values) == 1:
                value = unique_values[0]
                column_attr = self.FILTER_ATTR[column_enum]
                setattr(
                    self.filter_options,
                    column_attr,
                    FilterOption(column_name=column_enum, value=value),
                )

                # Update button state
                for button in self.button_groups[column_enum]:
                    if button.description == str(value):
                        button.value = True
                        break

        # Update button states based on filtered data
        self.update_button_states(filtered_df)

In [ ]:
class WorkflowResults(BaseModel):
    pass_rate_percent: float
    evaluated_pixel_count: int
    passed_pixel_count: int
    failed_pixel_count: int

    gamma_image_path: tp.Optional[Path]
    failure_image_path: tp.Optional[Path]
    registered_overlay_path: tp.Optional[Path]
    loaded_images_path: tp.Optional[Path]
    reference_image_path: tp.Optional[Path]
    measured_image_path: tp.Optional[Path]
    aligned_measured_path: tp.Optional[Path]

    measured_pssar: float
    reference_pssar: float
    scaling_error: float

    dose_to_agreement: float
    distance_to_agreement: float

    # --- Validators ---

    @validator(
        "gamma_image_path",
        "failure_image_path",
        "registered_overlay_path",
        "loaded_images_path",
        "reference_image_path",
        "measured_image_path",
        "aligned_measured_path",
        pre=True,
    )
    def convert_str_to_path(cls, v: tp.Optional[str]) -> tp.Optional[Path]:
        if v is None:
            return v
        return Path(v)

    @validator("pass_rate_percent")
    def validate_pass_rate(cls, v: float) -> float:
        if not (0.0 <= v <= 100.0):
            raise ValueError("pass_rate_percent must be between 0 and 100")
        return v

    @validator("evaluated_pixel_count", "passed_pixel_count", "failed_pixel_count")
    def validate_counts(cls, v: int) -> int:
        if v < 0:
            raise ValueError("pixel counts must be non-negative")
        return v

    @validator("scaling_error")
    def validate_scaling_error(cls, v: float) -> float:
        if not isinstance(v, float):
            raise ValueError("scaling_error must be a float")
        return v

    def __str__(self):
        # Prepare key-value pairs as strings
        items: list[str] = []
        max_len = max(len(k) for k in self.__dict__)
        for k, v in self.__dict__.items():
            v_str = str(v) if isinstance(v, Path) else repr(v)
            items.append(f"{k.ljust(max_len)} : {v_str}")
        return "\n".join(items)

    def save_to_json(self, path: tp.Union[str, Path]) -> None:
        """Save WorkflowResults to JSON file."""
        file_path = Path(path)
        file_path.parent.mkdir(exist_ok=True, parents=True)

        # Convert to dict and handle Path objects
        data = self.dict()
        for key, value in data.items():
            if isinstance(value, Path):
                data[key] = str(value)

        with open(file_path, "w") as f:
            json.dump(data, f, indent=2)

    @classmethod
    def load_from_json(cls, path: tp.Union[str, Path]) -> "WorkflowResults":
        """Load WorkflowResults from JSON file."""
        with open(path) as f:
            data = json.load(f)
        return cls(**data)

    class Config:
        arbitrary_types_allowed = True

In [ ]:
TRANSPARENT_PNG = b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00\x01\x08\x06\x00\x00\x00\x1f\x15\xc4\x89\x00\x00\x00\nIDATx\x9cc`\x00\x00\x00\x02\x00\x01\xe2!\xbc3\x00\x00\x00\x00IEND\xaeB`\x82"


def resample_colorbar_to_match_plot_inplace(colorbar_path: str, plot_path: str) -> None:
    """
    Resize a colorbar image so its height matches the plot image height.
    The colorbar image is overwritten in-place.

    Args:
        colorbar_path (str): Path to the colorbar image (will be overwritten)
        plot_path (str): Path to the reference plot image
    """
    # Get target height from plot
    with Image.open(plot_path) as plot_img:
        target_height = plot_img.size[1]

    # Load and resize colorbar
    with Image.open(colorbar_path) as cb_img:
        cb_img = cb_img.convert("RGBA")  # ensure consistent format
        cb_width, cb_height = cb_img.size

        if cb_height == 0:
            raise ValueError("Colorbar height is zero, cannot resample.")

        # Preserve aspect ratio
        new_width = int(cb_width * (target_height / cb_height))

        resized_cb = cb_img.resize((new_width, target_height), Image.LANCZOS)

        # Overwrite original file
        resized_cb.save(colorbar_path, format="PNG")


def make_transparent_png(width: int, height: int) -> bytes:
    img = Image.new("RGBA", (width, height), (0, 0, 0, 0))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def placeholder_from_png(path: str) -> bytes:
    with Image.open(path) as img:
        w, h = img.size
    return make_transparent_png(w, h)


class GuiColors(Enum):
    PRIMARY = "#0090D0"
    WHITE = "#FFFFFF"
    LOADING = "#566670"
    FAIL = "#9B2423"
    TEXT_PRIMARY = "#FFFFFF"


class ResultTableRow(Enum):
    SSAR = "sSAR [W/kg]"


class ResultTableColumn(Enum):
    REF_30_DBM = "Reference 30 dBm"
    MEASURED_30_DBM = "Measured 30 dBm"
    SCALING_ERROR = "Scaling Error [%]"


class SarGammaComparisonUI:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        self.logging_window = OutputWidgetHandler()
        self.logging_window.setFormatter(
            logging.Formatter("%(asctime)s  - [%(levelname)s] %(message)s")
        )
        self.logger.addHandler(self.logging_window)
        self.logger.setLevel(logging.INFO)

        self.simulated_files_df = SimulatedFilesDB()
        self.radio_button_grid = RadioButtonGrid(
            simulated_files_db=self.simulated_files_df
        )

        self.workflow_results: WorkflowResults

        # Thread control
        self._progress_thread = None
        self._stop_event = threading.Event()

        self.settings_checker_thread = threading.Thread(
            target=self.check_settings, daemon=True
        )
        self.settings_checker_thread.start()

        display(self.create_ui())

        # Attempt to restore state if previous session files exist
        self.restore_state()

    # ---------------------------
    #  Progress bar helper logic
    # ---------------------------
    def _start_progress_updater(self):
        """Starts a thread that gradually increases progress up to ~90%."""
        self._stop_event.clear()

        # Show progress bar
        with self.progress_output:
            self.progress_output.clear_output()
            display(self.progress_bar)

        def update_progress():
            duration = 240
            interval = 0.1
            steps = int(duration / interval)
            for i in range(steps):
                if self._stop_event.is_set():
                    break
                progress = min(0.9, i / steps * 0.9)
                self.progress_bar.value = progress
                time.sleep(interval)

        self.progress_bar.value = 0.0
        self.progress_bar.bar_style = "info"
        self.progress_bar.style = {"bar_color": GuiColors.PRIMARY.value}

        self._progress_thread = threading.Thread(target=update_progress, daemon=True)
        self._progress_thread.start()

    def _stop_progress_updater(self, completed: bool = False):
        self._stop_event.set()

        if self._progress_thread and self._progress_thread.is_alive():
            self._progress_thread.join(timeout=1.0)

        if completed:
            for v in range(int(self.progress_bar.value * 100), 101, 5):
                self.progress_bar.value = v / 100
                time.sleep(0.05)
            self.progress_bar.value = self.progress_bar.max

        self.progress_bar.style = (
            {"bar_color": GuiColors.PRIMARY.value}
            if completed
            else {"bar_color": GuiColors.FAIL.value}
        )

        # Hide progress bar
        self.progress_output.clear_output()

    def check_settings(self):
        interval = 1.0
        while True:
            try:
                elements_in_filtered_db = len(self.radio_button_grid.filtered_db)
                if (
                    elements_in_filtered_db == 1
                    and Path(FilePaths.MEASURED_FILE_PATH.value).exists()
                ):
                    self.run_button.disabled = False
                else:
                    self.run_button.disabled = True
            except AttributeError:
                pass
            time.sleep(interval)

    # ---------------------------
    #  Core button logic
    # ---------------------------
    def handle_button_click(self, button: widgets.Button):
        self.logger.debug("Start SAR Pattern Validation...")
        button.style = dict(
            button_color=GuiColors.LOADING.value,
            text_color=GuiColors.TEXT_PRIMARY.value,
        )
        button.disabled = True
        self.progress_bar.value = 0.0
        self.progress_bar.bar_style = "info"

        self._start_progress_updater()

        try:
            workspace_var = os.getenv("DY_SIDECAR_STATE_PATHS")
            workspace_var = workspace_var[2:-2] if workspace_var else ""
            self.logger.debug(
                self.radio_button_grid.filtered_db[
                    DBColumnNames.FILE_PATH.value
                ].to_list()[0]
            )

            run_output = run_sarsample(
                "--measured_file_path",
                f"{FilePaths.MEASURED_FILE_PATH.value}",
                "--reference_file_path",
                f"{self.radio_button_grid.filtered_db[DBColumnNames.FILE_PATH.value].to_list()[0]}",
                "--reference_image_save_path",
                f"{FilePaths.REFERENCE_IMAGE_PATH.value}",
                "--measured_image_save_path",
                f"{FilePaths.MEASURED_IMAGE_PATH.value}",
                "--aligned_meas_save_path",
                f"{FilePaths.ALIGNED_MEANS_PATH.value}",
                "--registered_image_save_path",
                f"{FilePaths.REGISTERED_IMAGE_PATH.value}",
                "--gamma_comparison_image_path",
                f"{FilePaths.GAMMA_COMPARISON_PATH.value}",
                "--power_level_dbm",
                f"{self.power_level.value}",
            )

            self.logger.info("SAR Pattern Validation done.")
            self.logger.debug(run_output)

            self.workflow_results = WorkflowResults(**run_output["result"])

            self._stop_progress_updater(completed=True)

            # Save workflow results and filtered DB
            self._save_workflow_state()

            # Update state.json with power level
            try:
                state_path = Path(FilePaths.STATE_JSON_PATH.value)
                if state_path.exists():
                    with open(state_path) as f:
                        state_data = json.load(f)
                    state_data["power_level"] = self.power_level.value
                    with open(state_path, "w") as f:
                        json.dump(state_data, f, indent=2)
            except Exception as e:
                self.logger.warning(f"Failed to save power level: {e}")

            self.update_images()
            self._update_analytical_results(sarsample_results=self.workflow_results)
        except Exception as e:
            button.style = dict(
                button_color=GuiColors.FAIL.value,
                text_color=GuiColors.TEXT_PRIMARY.value,
            )
            self._stop_progress_updater(completed=False)
            self.logger.warning(e)
        finally:
            button.style = dict(
                button_color=GuiColors.PRIMARY.value,
                text_color=GuiColors.TEXT_PRIMARY.value,
            )
            button.disabled = False

    def _update_analytical_results(self, sarsample_results: WorkflowResults):

        self.logger.info(sarsample_results)
        self.logger.debug(type(sarsample_results))

        passed = (
            sarsample_results.passed_pixel_count
            == sarsample_results.evaluated_pixel_count
        )
        pass_rate = sarsample_results.pass_rate_percent
        measured_pssar = sarsample_results.measured_pssar
        reference_pssar = sarsample_results.reference_pssar
        scaling_error = sarsample_results.scaling_error

        if passed:
            self.result_indicator_button.description = "Pass"
            self.result_indicator_button.style = dict(
                button_color=GuiColors.PRIMARY.value,
                text_color=GuiColors.TEXT_PRIMARY.value,
            )
        else:
            self.result_indicator_button.description = "Fail"
            self.result_indicator_button.style = dict(
                button_color=GuiColors.FAIL.value,
                text_color=GuiColors.TEXT_PRIMARY.value,
            )

        self.pass_rate_label.value = f"<b>Pass rate = {pass_rate:.1f}%</b>"

        values = [
            f"{reference_pssar:.2f}",
            f"{measured_pssar:.2f}",
            f"{100 * scaling_error:.2f}",
        ]

        table_html = f"""
        <table style="border-collapse: collapse; width: 100%; font-family: Arial;">
            <thead>
                <tr style="background-color: #f2f2f2;">
                    <th style="border: 1px solid #ddd; padding: 8px;"></th>
                    {"".join(f'<th style="border: 1px solid #ddd; padding: 8px;">{col.value}</th>' for col in ResultTableColumn)}
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style="border: 1px solid #ddd; padding: 8px;"><b>{ResultTableRow.SSAR.value}</b></td>
                    {"".join(f'<td style="border: 1px solid #ddd; padding: 8px;">{val}</td>' for val in values)}
                </tr>
            </tbody>
        </table>
        """

        self.result_table.value = table_html

    def _on_file_upload_change(self, change: Bunch):
        """Triggered immediately when a file is selected."""
        try:
            value = change["new"]

            if not value:
                self.uploaded_file_name_label.value = ""
                return

            file_info = value[0]
            name = file_info["name"]

            # Update label immediately
            self.uploaded_file_name_label.value = str(name)

            # Save immediately
            self.save_uploaded_file(
                upload_widget=self.upload_1,
                save_path=FilePaths.MEASURED_FILE_PATH.value,
            )

            # Save state with measured file name
            self._save_measured_file_state(name)

            self.logger.debug(f"File selected: {name}")

        except Exception as e:
            self.logger.warning(e)

    def save_uploaded_file(self, upload_widget: widgets.FileUpload, save_path: str):
        try:
            if not upload_widget.value:
                return

            file_info: dict[str, tp.Any] = upload_widget.value[0]
            name = file_info["name"]
            content = file_info["content"]

            file_path = Path(save_path)
            file_path.parent.mkdir(exist_ok=True, parents=True)

            with open(save_path, "wb") as f:
                f.write(content)

            self.logger.debug(f"Saved {name} → {save_path}")

        except Exception as e:
            self.logger.warning(e)

    def _save_measured_file_state(self, filename: str) -> None:
        """Save the measured file name to state JSON."""
        try:
            state_path = Path(FilePaths.STATE_JSON_PATH.value)
            state_path.parent.mkdir(exist_ok=True, parents=True)

            state_data: dict[str, tp.Any] = {
                "measured_file_name": filename,
                "timestamp": time.time(),
            }

            with open(state_path, "w") as f:
                json.dump(state_data, f, indent=2)

            self.logger.debug(f"Saved state: {filename}")
        except Exception as e:
            self.logger.warning(f"Failed to save measured file state: {e}")

    def _save_workflow_state(self) -> None:
        """Save workflow results and filtered DB."""
        try:
            # Save workflow results
            self.workflow_results.save_to_json(
                FilePaths.WORKFLOW_RESULTS_JSON_PATH.value
            )

            # Save filtered DB
            self.simulated_files_df.save_filtered_db_to_csv(
                self.radio_button_grid.filtered_db, FilePaths.FILTERED_DB_CSV_PATH.value
            )

            self.logger.debug("Saved workflow state")
        except Exception as e:
            self.logger.warning(f"Failed to save workflow state: {e}")

    def _check_restore_files_exist(self) -> bool:
        """Check if all necessary files exist for restore."""
        required_files = [
            FilePaths.MEASURED_FILE_PATH.value,
            FilePaths.STATE_JSON_PATH.value,
            FilePaths.WORKFLOW_RESULTS_JSON_PATH.value,
            FilePaths.FILTERED_DB_CSV_PATH.value,
            FilePaths.REFERENCE_IMAGE_PATH.value,
            FilePaths.MEASURED_IMAGE_PATH.value,
            FilePaths.ALIGNED_MEANS_PATH.value,
            FilePaths.REGISTERED_IMAGE_PATH.value,
            FilePaths.GAMMA_COMPARISON_PATH.value,
            FilePaths.GAMMA_COMPARISON_FAILURES_PATH.value,
        ]

        return all(Path(f).exists() for f in required_files)

    def restore_state(self) -> None:
        """Restore GUI state from saved files."""
        try:
            if not self._check_restore_files_exist():
                self.logger.debug("Not all restore files exist, skipping restore")
                return

            self.logger.info("Restoring previous session state...")

            # Load state JSON
            with open(FilePaths.STATE_JSON_PATH.value) as f:
                state_data = json.load(f)
            measured_file_name = state_data.get("measured_file_name", "")
            power_level = state_data.get("power_level", 23.0)

            # Load workflow results
            self.workflow_results = WorkflowResults.load_from_json(
                FilePaths.WORKFLOW_RESULTS_JSON_PATH.value
            )

            # Load filtered DB
            filtered_db = self.simulated_files_df.load_filtered_db_from_csv(
                FilePaths.FILTERED_DB_CSV_PATH.value
            )

            # Restore radio button grid state
            self.radio_button_grid.restore_from_filtered_db(filtered_db)

            # Restore GUI state
            self.uploaded_file_name_label.value = measured_file_name
            self.power_level.value = power_level

            # Update images
            self.update_images(no_data=False)

            # Update analytical results
            self._update_analytical_results(sarsample_results=self.workflow_results)
            self.radio_button_grid.restore_from_filtered_db(filtered_db)
            self.logger.info("Session state restored successfully")

        except Exception as e:
            self.logger.warning(f"Failed to restore state: {e}")
            # If restore fails, clear any partial state and start fresh
            self.update_images(no_data=True)

    def update_images(self, no_data: bool = False):
        if no_data:
            image_files = [FilePaths.NO_DATA_IMAGE.value] * 8
        else:
            resample_colorbar_to_match_plot_inplace(
                colorbar_path=FilePaths.ALIGNED_MEANS_COLORBAR_PATH.value,
                plot_path=FilePaths.ALIGNED_MEANS_PATH.value,
            )
            resample_colorbar_to_match_plot_inplace(
                colorbar_path=FilePaths.GAMMA_COMPARISON_COLORBAR_PATH.value,
                plot_path=FilePaths.GAMMA_COMPARISON_PATH.value,
            )
            image_files = [
                FilePaths.REFERENCE_IMAGE_PATH.value,
                FilePaths.MEASURED_IMAGE_PATH.value,
                FilePaths.ALIGNED_MEANS_PATH.value,
                FilePaths.ALIGNED_MEANS_COLORBAR_PATH.value,
                FilePaths.REGISTERED_IMAGE_PATH.value,
                FilePaths.GAMMA_COMPARISON_PATH.value,
                FilePaths.GAMMA_COMPARISON_COLORBAR_PATH.value,
                FilePaths.GAMMA_COMPARISON_FAILURES_PATH.value,
            ]
        widgets_list = [
            self.image_top_left,
            self.image_top_middle,
            self.image_top_right,
            self.colorbar_top,
            self.image_bottom_left,
            self.image_bottom_middle,
            self.colorbar_bottom,
            self.image_bottom_right,
        ]

        if len(widgets_list) != len(image_files):
            raise ValueError("widgets_list and image_files must have the same length")

        for img_widget, path in zip(widgets_list, image_files):
            if no_data:
                img_widget.value = TRANSPARENT_PNG
            else:
                try:
                    if isinstance(path, str):
                        with open(path, "rb") as f:
                            img_widget.value = f.read()
                    else:
                        img_widget.value = path
                except FileNotFoundError:
                    pass

        # Update placeholders (separate instances, each needs its own data)
        if no_data:
            ph_data = TRANSPARENT_PNG
        else:
            ph_data = placeholder_from_png(
                path=FilePaths.ALIGNED_MEANS_COLORBAR_PATH.value
            )
        for ph in [
            self.placeholder_1,
            self.placeholder_2,
            self.placeholder_3,
            self.placeholder_4,
        ]:
            ph.value = ph_data

    def create_maintenance_ui(self) -> widgets.HTML:
        """
        Creates a stylized maintenance / access-restricted UI message.
        """
        return widgets.HTML(
            value="""
            <div style="
                display: flex;
                align-items: center;
                justify-content: center;
                width: 100%;
                padding: 40px 20px;
            ">
                <div style="
                    background-color: #F4FAFD;
                    border: 1px solid #CDE7F5;
                    border-left: 6px solid #0090D0;
                    border-radius: 10px;
                    padding: 24px 28px;
                    max-width: 700px;
                    box-shadow: 0 4px 10px rgba(0, 0, 0, 0.05);
                    font-family: Arial, sans-serif;
                    color: #003B5C;
                ">
                    <div style="
                        font-size: 18px;
                        font-weight: 600;
                        margin-bottom: 8px;
                    ">
                        🔒 SAR Pattern Validation
                    </div>
                    <div style="
                        font-size: 15px;
                        line-height: 1.5;
                    ">
                        SAR pattern validation is currently restricted.
                        <br />
                        Contact
                        <a href="mailto:support@itis.swiss"
                           style="color: #0090D0; text-decoration: none; font-weight: 500;">
                            support@itis.swiss
                        </a>
                        for access.
                    </div>
                </div>
            </div>
            """,
            layout=widgets.Layout(width="100%"),
        )

    def create_ui(
        self,
        left_ratio: float = 0.3,
        right_ratio: float = 0.7,
        side_gap: str = "100px",
    ) -> widgets.VBox:
        # ---------- Helper Layout Functions ----------

        def row(children, gap: str = "10px", align: str = "center") -> widgets.HBox:
            return widgets.HBox(
                children=children,
                layout=widgets.Layout(
                    gap=gap,
                    width="100%",
                    align_items=align,
                    overflow="hidden",
                ),
            )

        def col(children, gap: str = "10px") -> widgets.VBox:
            return widgets.VBox(
                children=children,
                layout=widgets.Layout(
                    gap=gap,
                    width="100%",
                    align_items="stretch",
                    overflow="hidden",
                ),
            )

        def flex_item(widget, flex="1", min_width="0"):
            widget.layout = widgets.Layout(
                flex=flex, min_width=min_width, overflow="hidden"
            )
            return widget

        # ---------- File Upload ----------

        self.upload_1 = widgets.FileUpload(
            accept=".csv",
            multiple=False,
            description="Measured CSV",
            style={
                "button_color": GuiColors.PRIMARY.value,
                "text_color": GuiColors.TEXT_PRIMARY.value,
            },
        )
        self.upload_1.observe(self._on_file_upload_change, names="value")

        self.power_level = widgets.BoundedFloatText(
            value=23.0,
            min=-10,
            max=50,
            step=0.1,
            description="power level [dBm]:",
            style={"description_width": "initial"},
        )

        top_row = row(
            [flex_item(self.upload_1, "1"), flex_item(self.power_level, "1", "150px")]
        )

        self.uploaded_file_name_label = widgets.Label(value="")

        tooltip = widgets.HTML(
            value="""
            <div style="
                background-color: #E8F6FD;
                color: #005A8C;
                border-left: 4px solid #0090D0;
                padding: 8px 12px;
                border-radius: 6px;
                font-size: 14px;
            ">
                <b>ℹ️ Note:</b> The uploaded <code>.csv</code> files must be smaller than <b>10 MB</b>.
            </div>
            """
        )

        radio_button_grid = self.radio_button_grid.create_radio_button_grid()

        left_setup_section = col(
            [top_row, self.uploaded_file_name_label, tooltip, radio_button_grid]
        )
        left_setup_section.layout.flex = str(left_ratio)

        # ---------- Results Section ----------

        self.run_button = widgets.Button(
            description="Compare Patterns",
            style=dict(
                button_color=GuiColors.PRIMARY.value,
                text_color=GuiColors.TEXT_PRIMARY.value,
            ),
            disabled=True,
        )
        self.run_button.on_click(self.handle_button_click)

        self.result_indicator_button = widgets.Button(
            description="",
            style=dict(
                button_color=GuiColors.WHITE.value,
                text_color=GuiColors.TEXT_PRIMARY.value,
            ),
        )

        self.pass_rate_label = widgets.HTML(value="")  # <-- change to HTML
        self.result_table = widgets.HTML(value="")

        # Group result elements tightly
        result_info_group = row(
            [
                flex_item(self.result_indicator_button, "0 0 80px"),
                flex_item(self.pass_rate_label, "0 0 auto"),
                flex_item(self.result_table, "2"),
            ],
            gap="10px",
        )

        # Main row: button separated from results
        results_top_section = row(
            [
                flex_item(self.run_button, "0 0 auto"),
                result_info_group,
            ],
            gap="40px",  # <-- creates visual separation
        )

        # ---------- Progress Bar ----------

        self.progress_bar = widgets.FloatProgress(
            value=0.0,
            min=0.0,
            max=1.0,
            description="Progress:",
            bar_style="info",
            style={"bar_color": GuiColors.PRIMARY.value},
            layout=widgets.Layout(width="95%"),
        )

        self.progress_output = widgets.Output(layout=widgets.Layout(width="100%"))

        progress_bar_container = row(
            [self.progress_output], gap="0px", align="flex-start"
        )

        # ---------- Images ----------

        def wrap_image(img: widgets.Image, flex: str = "1") -> widgets.Box:
            return widgets.Box(
                [img],
                layout=widgets.Layout(
                    flex=flex,
                    height="100%",
                    overflow="hidden",
                    align_items="stretch",
                    justify_content="center",
                ),
            )

        def create_main_image() -> tuple[widgets.Image, widgets.Box]:
            img = widgets.Image(
                format="png",
                layout=widgets.Layout(
                    width="100%",
                    height="100%",
                    object_fit="contain",
                ),
            )
            return img, wrap_image(img, flex="4")

        def create_colorbar_image() -> tuple[widgets.Image, widgets.Box]:
            img = widgets.Image(
                format="png",
                layout=widgets.Layout(
                    width="100%",
                    height="100%",
                    object_fit="contain",
                ),
            )
            return img, wrap_image(img, flex="1")

        # Top row plots
        self.image_top_left, box_top_left = create_main_image()
        self.image_top_middle, box_top_middle = create_main_image()
        self.image_top_right, box_top_right = create_main_image()

        # Bottom row plots
        self.image_bottom_left, box_bottom_left = create_main_image()
        self.image_bottom_middle, box_bottom_middle = create_main_image()
        self.image_bottom_right, box_bottom_right = create_main_image()

        # Colorbars
        self.colorbar_top, box_cb_top = create_colorbar_image()
        self.colorbar_bottom, box_cb_bottom = create_colorbar_image()

        # Placeholders (each slot needs its own widget instance)
        self.placeholder_1, box_ph_1 = create_colorbar_image()
        self.placeholder_2, box_ph_2 = create_colorbar_image()
        self.placeholder_3, box_ph_3 = create_colorbar_image()
        self.placeholder_4, box_ph_4 = create_colorbar_image()

        self.update_images(no_data=True)

        # -----------------------------------
        # Build the grid
        # -----------------------------------

        row_layout = widgets.Layout(
            width="100%", align_items="stretch", justify_content="space-between"
        )

        top_row = widgets.HBox(
            [
                widgets.HBox(
                    [box_top_left, box_ph_1],
                    layout=widgets.Layout(width="33%", align_items="stretch"),
                ),
                widgets.HBox(
                    [box_top_middle, box_ph_2],
                    layout=widgets.Layout(width="33%", align_items="stretch"),
                ),
                widgets.HBox(
                    [box_top_right, box_cb_top],
                    layout=widgets.Layout(width="33%", align_items="stretch"),
                ),
            ],
            layout=row_layout,
        )

        bottom_row = widgets.HBox(
            [
                widgets.HBox(
                    [box_bottom_left, box_ph_3],
                    layout=widgets.Layout(width="33%", align_items="stretch"),
                ),
                widgets.HBox(
                    [box_bottom_middle, box_cb_bottom],
                    layout=widgets.Layout(width="33%", align_items="stretch"),
                ),
                widgets.HBox(
                    [box_bottom_right, box_ph_4],
                    layout=widgets.Layout(width="33%", align_items="stretch"),
                ),
            ],
            layout=row_layout,
        )

        grid = widgets.VBox([top_row, bottom_row], layout=widgets.Layout(width="100%"))

        images_section = widgets.Box(
            [grid],
            layout=widgets.Layout(
                max_height="600px",  # constrain height
                overflow_y="auto",  # single scrollbar
                padding="5px",
            ),
        )

        right_results_section = col(
            [
                results_top_section,
                progress_bar_container,
                images_section,
            ]
        )
        right_results_section.layout.flex = str(right_ratio)

        # ---------- Main Layout ----------

        main_gui_section = widgets.HBox(
            [left_setup_section, right_results_section],
            layout=widgets.Layout(gap=side_gap, width="100%", align_items="flex-start"),
        )

        # ---------- Logging ----------

        log_box = self.logging_window.show_logs()

        return widgets.VBox(
            [main_gui_section, log_box],
            layout=widgets.Layout(gap="10px", width="100%", align_items="flex-start"),
        )

In [ ]:
sar_gamma_comparison_ui = SarGammaComparisonUI()